# BridgeWatch AI: Model Training

Project: BridgeWatch AI, predicting whether a bridge's condition will deteriorate by its next inspection.
Author: Edmund Goldsberry, UMBC DATA 606 Capstone.

The model uses three data sources: bridge inspection records, weather at each bridge's location, and each
state's spending on bridge repair.

## 1. Load and join the bridge, weather, and spending data

* `data/processed/bridge_deterioration_dataset.csv.gz`: one row per bridge per year to year transition
  (2020 to 21 through 2024 to 25), with the `deteriorated_next_period` label. A single gzip compressed file.
* `data/weather_annual_by_bridge_2020.csv` through `data/weather_annual_by_bridge_2025.csv`: one row per
  bridge per calendar year, with annual temperature and precipitation summaries from NOAA. Split into six
  plain csv files, one per year, so each file stays under GitHub's per file size limits.
* `data/sf12a_state_bridge_spending_2020_2024.csv`: one row per state per year (2020 to 2024), with what each
  state spent on bridge replacement and repair, in thousands of dollars, from FHWA Table SF-12A.

The weather data joins on the bridge ID and year. The spending data only exists at the state level, so it joins
on the state and year instead, and every bridge in the same state and year gets the same value.

Raw spending mostly tells you how big a state is (Texas has about ten times as many bridges as Maryland), so the
spending is turned into repair spending per bridge: the state's repair total divided by its number of
bridges that year.

In [1]:
import sys
!{sys.executable} -m pip install pandas numpy scikit-learn plotly joblib streamlit
import sys
print(sys.executable)
print(sys.path)
import os
import urllib.error
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/goldiemonster/UMBC-DATA606-Capstone/main"
LOCAL_DATA_DIR = os.path.join("..", "data")
KEY = "STRUCTURE_NUMBER_008"
WEATHER_YEARS = range(2020, 2026)


def load_bridge_data():
    """Load the bridge-year-transition dataset built in eda.ipynb.

    Tries the GitHub repo first, and falls back to a local copy if that fails.
    """
    remote_path = "data/processed/bridge_deterioration_dataset.csv.gz"
    local_path = os.path.join(LOCAL_DATA_DIR, "processed", "bridge_deterioration_dataset.csv.gz")
    try:
        df = pd.read_csv(f"{GITHUB_RAW_BASE}/{remote_path}", compression="gzip")
        print(f"Loaded {remote_path} from GitHub")
        return df
    except (urllib.error.URLError, urllib.error.HTTPError, OSError) as exc:
        print(f"Could not load {remote_path} from GitHub ({exc}); falling back to {local_path}")
        return pd.read_csv(local_path, compression="gzip")


def load_weather_data():
    """Load the annual per-bridge weather dataset, split into one csv per year on GitHub.

    Tries to fetch all six yearly files from GitHub. If any single file is missing, the whole
    attempt is abandoned and a local, single gzip copy of the full dataset is used instead,
    so a partial GitHub fetch never gets silently mixed with local data.
    """
    try:
        frames = []
        for year in WEATHER_YEARS:
            remote_path = f"data/weather_annual_by_bridge_{year}.csv"
            frames.append(pd.read_csv(f"{GITHUB_RAW_BASE}/{remote_path}"))
        print(f"Loaded weather_annual_by_bridge_{{{min(WEATHER_YEARS)}..{max(WEATHER_YEARS)}}}.csv from GitHub")
        return pd.concat(frames, ignore_index=True)
    except (urllib.error.URLError, urllib.error.HTTPError, OSError) as exc:
        local_path = os.path.join(LOCAL_DATA_DIR, "processed", "weather_annual_by_bridge.csv.gz")
        print(f"Could not load all weather years from GitHub ({exc}); falling back to {local_path}")
        return pd.read_csv(local_path, compression="gzip")


def join_bridge_and_weather(bridge_df, weather_df):
    """Left join weather onto bridge_df, matching (KEY, from_year) to (KEY, YEAR)."""
    weather_renamed = weather_df.drop(columns="STATE").rename(columns={"YEAR": "weather_year"})
    merged = bridge_df.merge(
        weather_renamed,
        left_on=[KEY, "from_year"],
        right_on=[KEY, "weather_year"],
        how="left",
    ).drop(columns="weather_year")
    return merged


def load_spending_data():
    """Load the state bridge spending file from GitHub, falling back to a local copy."""
    remote_path = "data/sf12a_state_bridge_spending_2020_2024.csv"
    local_path = os.path.join(LOCAL_DATA_DIR, "sf12a_state_bridge_spending_2020_2024.csv")
    try:
        df = pd.read_csv(f"{GITHUB_RAW_BASE}/{remote_path}")
        print(f"Loaded {remote_path} from GitHub")
        return df
    except (urllib.error.URLError, urllib.error.HTTPError, OSError) as exc:
        print(f"Could not load {remote_path} from GitHub ({exc}); falling back to {local_path}")
        return pd.read_csv(local_path)


def spending_per_bridge(spending_df, bridge_df):
    """Divide each state's yearly repair total by its number of bridges that year."""
    counts = bridge_df.groupby(["STATE", "from_year"]).size().rename("n_bridges").reset_index()
    spending = spending_df.rename(columns={"state": "STATE", "year": "from_year"})
    spending = spending.merge(counts, on=["STATE", "from_year"], how="left")
    spending["repair_per_bridge_thousands"] = spending["bridge_repair_total_thousands"] / spending["n_bridges"]
    return spending[["STATE", "from_year", "repair_per_bridge_thousands"]]


def join_spending(df, spending_df):
    """Left join the per-bridge spending onto df, matching on STATE and from_year."""
    return df.merge(spending_df, on=["STATE", "from_year"], how="left")

/usr/local/bin/python3
['/usr/lib/python311.zip', '/usr/lib/python3.11', '/usr/lib/python3.11/lib-dynload', '', '/root/.local/lib/python3.11/site-packages', '/usr/local/lib/python3.11/dist-packages', '/usr/lib/python3/dist-packages']


In [2]:
bridge_df = load_bridge_data()
weather_df = load_weather_data()

print(f"Bridge dataset: {bridge_df.shape[0]:,} rows x {bridge_df.shape[1]} columns")
print(f"Weather dataset: {weather_df.shape[0]:,} rows x {weather_df.shape[1]} columns")

df = join_bridge_and_weather(bridge_df, weather_df)

assert len(df) == len(bridge_df), "Join should not change the row count (weather is one row per bridge-year)"
weather_match_rate = df["tavg_mean_c"].notna().mean() * 100

print(f"\nJoined dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Weather match rate: {weather_match_rate:.1f}%")
df.head()

Loaded data/processed/bridge_deterioration_dataset.csv.gz from GitHub


Loaded weather_annual_by_bridge_{2020..2025}.csv from GitHub
Bridge dataset: 681,841 rows x 39 columns
Weather dataset: 841,608 rows x 8 columns



Joined dataset: 681,841 rows x 44 columns
Weather match rate: 98.5%


,STRUCTURE_NUMBER_008,COUNTY_CODE_003,LAT_016,LONG_017,OWNER_022,FUNCTIONAL_CLASS_026,YEAR_BUILT_027,TRAFFIC_LANES_ON_028A,ADT_029,YEAR_ADT_030,STRUCTURE_KIND_043A,STRUCTURE_TYPE_043B,MAIN_UNIT_SPANS_045,APPR_SPANS_046,MAX_SPAN_LEN_MT_048,STRUCTURE_LEN_MT_049,DECK_WIDTH_MT_052,DECK_COND_058,SUPERSTRUCTURE_COND_059,SUBSTRUCTURE_COND_060,CULVERT_COND_062,DATE_OF_INSPECT_090,INSPECT_FREQ_MONTHS_091,DECK_STRUCTURE_TYPE_107,PERCENT_ADT_TRUCK_109,SCOUR_CRITICAL_113,BRIDGE_CONDITION,LOWEST_RATING,STATE,YEAR,LOWEST_RATING_next,BRIDGE_CONDITION_next,from_year,to_year,deteriorated_next_period,age,lat,lon,months_since_inspection,tmax_mean_c,tmin_mean_c,tavg_mean_c,prcp_total_mm,freeze_thaw_days
0,06 0021,89,40454199.0,122197000.0,69,1,1941,4,19500,2009.0,3,9,8,7,192.0,1093.6,18.9,5,7,7,N,518,24,1,29.0,8,F,5,CA,2020,5,F,2020,2021,0,79,40.761664,-122.336111,24,25.256404,9.759584,17.508389,682.585938,3.0
1,19P0005,61,39000000.0,120562375.0,69,9,1935,1,75,2017.0,3,10,1,3,39.6,67.7,4.0,5,6,3,N,518,24,8,NaN,2,P,3,CA,2020,3,P,2020,2021,0,85,39.000000,-120.939931,24,23.319907,8.978441,16.149441,653.156250,17.0
2,1CA0070,111,34060810.0,119061488.0,73,9,1949,2,200,2013.0,1,4,1,0,10.4,10.4,17.2,6,5,6,N,819,24,2,2.0,5,F,5,CA,2020,5,F,2020,2021,0,71,34.102250,-119.104133,24,20.630550,12.516244,16.572938,167.789062,0.0
3,1CA0095,73,32433594.0,117084854.0,73,19,1985,2,100,2015.0,2,2,5,0,18.9,60.8,10.7,6,7,8,N,819,24,1,20.0,N,F,6,CA,2020,6,F,2020,2021,0,35,32.726650,-117.146817,24,22.369002,13.666090,18.017525,195.328125,0.0
4,1CA0141,111,34065929.0,119054787.0,73,9,1948,2,200,2013.0,1,5,4,0,6.9,27.4,15.9,7,7,5,N,819,24,2,2.0,5,F,5,CA,2020,5,F,2020,2021,0,72,34.116469,-119.096631,24,20.630550,12.516244,16.572938,167.789062,0.0


Next, add the spending data. The spending year is matched to `from_year`, the year each row's features come
from, so a bridge's 2022 record gets its state's 2022 spending. The spending file covers 2020 to 2024, which is
exactly the range of `from_year`, so every row should find a match.

In [3]:
spending_df = load_spending_data()
per_bridge = spending_per_bridge(spending_df, bridge_df)

rows_before = len(df)
df = join_spending(df, per_bridge)

assert len(df) == rows_before, "Join should not change the row count (spending is one row per state-year)"
spending_match_rate = df["repair_per_bridge_thousands"].notna().mean() * 100
print(f"Spending match rate: {spending_match_rate:.1f}%")

print("\nRepair spending per bridge, thousands of dollars:")
print(per_bridge.pivot(index="STATE", columns="from_year", values="repair_per_bridge_thousands").round(1))

Loaded data/sf12a_state_bridge_spending_2020_2024.csv from GitHub
Spending match rate: 100.0%

Repair spending per bridge, thousands of dollars:
from_year  2020  2021  2022  2023  2024
STATE                                  
CA          7.7   9.0   9.3  14.3  15.9
FL         24.5  33.4  41.8  41.8  39.7
MD         35.2  29.4  22.9  35.1  24.8
MI         51.4  52.9  56.1  53.3  66.9
NY         61.7  49.8  51.2  51.1  38.5
TX         14.7  14.1  14.2  22.6  25.8
WA         26.2  56.6  75.7  33.5  37.4


## 2. Define and clean the feature set

The model predicts `deteriorated_next_period` (1 if a bridge's condition rating dropped by the next
inspection, 0 otherwise). The candidate features fall into two groups:

* Categorical columns, encoded as one-hot indicators (material type, ownership, condition ratings, and so
  on). These get cast to text first, since a couple of them mix numeric-looking codes with the letter `N`
  for "not applicable", and mixing types can confuse the encoder later.
* Numeric columns, which get scaled before modeling (age, traffic volume, span length, the weather
  features, and repair spending per bridge, among others).

A few columns are left out on purpose: `STRUCTURE_NUMBER_008` is an identifier, not a predictor.
`YEAR_BUILT_027` is replaced by `age`, which is the more directly useful version of the same information.
`BRIDGE_CONDITION` and the year columns (`from_year`, `to_year`, `YEAR`) are either derived from the label or
used only for the train/test split, not fed to the model.

In [4]:
TARGET = "deteriorated_next_period"

CATEGORICAL_COLS = [
    "STRUCTURE_KIND_043A", "STRUCTURE_TYPE_043B", "SCOUR_CRITICAL_113", "OWNER_022", "STATE",
    "DECK_STRUCTURE_TYPE_107", "DECK_COND_058", "SUPERSTRUCTURE_COND_059", "SUBSTRUCTURE_COND_060",
    "CULVERT_COND_062",
]

NUMERIC_COLS = [
    "age", "ADT_029", "PERCENT_ADT_TRUCK_109", "MAIN_UNIT_SPANS_045", "STRUCTURE_LEN_MT_049",
    "DECK_WIDTH_MT_052", "TRAFFIC_LANES_ON_028A", "INSPECT_FREQ_MONTHS_091", "LOWEST_RATING", "lat", "lon",
    "tmax_mean_c", "tmin_mean_c", "tavg_mean_c", "prcp_total_mm", "freeze_thaw_days",
    "repair_per_bridge_thousands",
]

FEATURE_COLS = CATEGORICAL_COLS + NUMERIC_COLS

print(f"{len(CATEGORICAL_COLS)} categorical columns, {len(NUMERIC_COLS)} numeric columns, "
      f"{len(FEATURE_COLS)} features total")

10 categorical columns, 17 numeric columns, 27 features total


In [5]:
def clean_features(df, categorical_cols=CATEGORICAL_COLS):
    """Cast categorical columns to text and turn blank strings into real missing values."""
    df = df.copy()
    for col in categorical_cols:
        df[col] = df[col].astype(str).replace("", np.nan)
    return df

In [6]:
df = clean_features(df)

print("Missing values by feature (percent):")
print((df[FEATURE_COLS].isna().mean() * 100).round(2).sort_values(ascending=False))

print(f"\nTarget balance: {df[TARGET].value_counts(normalize=True).round(4).to_dict()}")

Missing values by feature (percent):
PERCENT_ADT_TRUCK_109          1.64
tavg_mean_c                    1.49
tmax_mean_c                    1.49
tmin_mean_c                    1.49
lon                            0.39
lat                            0.15
prcp_total_mm                  0.14
freeze_thaw_days               0.14
STRUCTURE_KIND_043A            0.00
STRUCTURE_TYPE_043B            0.00
SCOUR_CRITICAL_113             0.00
age                            0.00
CULVERT_COND_062               0.00
SUBSTRUCTURE_COND_060          0.00
SUPERSTRUCTURE_COND_059        0.00
DECK_COND_058                  0.00
DECK_STRUCTURE_TYPE_107        0.00
STATE                          0.00
OWNER_022                      0.00
LOWEST_RATING                  0.00
INSPECT_FREQ_MONTHS_091        0.00
TRAFFIC_LANES_ON_028A          0.00
DECK_WIDTH_MT_052              0.00
STRUCTURE_LEN_MT_049           0.00
MAIN_UNIT_SPANS_045            0.00
ADT_029                        0.00
repair_per_bridge_thousands

## 3. Split into train and test sets

The split is based on time, not a random shuffle. The model trains on transitions ending in 2021 through
2024, and gets evaluated on the transitions ending in 2025, which it never sees during training. This mirrors
how the model would actually be used: forecasting next year's risk for bridges that are already being
tracked, using only information that would have been available before that forecast.

The same bridge can show up in both the training set (its 2022 to 23 transition, for example) and the test
set (its 2024 to 25 transition). That is not label leakage. The model never sees a bridge's 2025 outcome while
training, since only earlier transitions are in the training set.

The spending data follows the same timeline. Training rows use spending from 2020 to 2023, and test rows use
2024 spending, which is the most recent year that would be known before the 2025 inspections.

In [7]:
def time_based_split(df, test_year=2025):
    """Train on every to_year before test_year; test on to_year == test_year."""
    train = df[df["to_year"] < test_year].copy()
    test = df[df["to_year"] == test_year].copy()
    return train, test

In [8]:
train_df, test_df = time_based_split(df)

assert len(train_df) + len(test_df) == len(df), "train and test should cover every row exactly once"
assert set(train_df["to_year"]).isdisjoint(set(test_df["to_year"])), "train and test years should not overlap"

print(f"Train: {len(train_df):,} rows, to_year in {sorted(train_df['to_year'].unique())}")
print(f"Test:  {len(test_df):,} rows, to_year in {sorted(test_df['to_year'].unique())}")

print(f"\nTrain positive rate: {100 * train_df[TARGET].mean():.2f}%")
print(f"Test positive rate:  {100 * test_df[TARGET].mean():.2f}% ({test_df[TARGET].sum():,} positives)")

print(f"\nSpending years used in train: {sorted(train_df['from_year'].unique())}")
print(f"Spending years used in test:  {sorted(test_df['from_year'].unique())}")

X_train, y_train = train_df[FEATURE_COLS], train_df[TARGET]
X_test, y_test = test_df[FEATURE_COLS], test_df[TARGET]

Train: 543,921 rows, to_year in [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Test:  137,920 rows, to_year in [np.int64(2025)]

Train positive rate: 4.29%
Test positive rate:  4.54% (6,268 positives)

Spending years used in train: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Spending years used in test:  [np.int64(2024)]


## 4. Build and fit the preprocessing and model pipeline

The model is a Logistic Regression, chosen for interpretability: its coefficients can be read directly as
"this factor pushes risk up" or "this factor pushes risk down", which is what the app needs to show for each
bridge.

Before the model can use the data, it needs two kinds of preprocessing:

* Numeric columns get missing values filled with the column median, then scaled to have a mean of 0 and a
  standard deviation of 1. Scaling matters for Logistic Regression because the size of each coefficient
  depends on the scale of its input; without it, a feature like prcp_total_mm (values in the hundreds)
  would dominate a feature like age (values in the tens) for reasons that have nothing to do with actual
  importance.
* Categorical columns get missing values filled with the most frequent category, then one hot encoded (turned
  into a set of 0/1 indicator columns, one per category).

Both steps are combined into a single scikit-learn pipeline, fit only on the training set, so nothing about
the test set leaks into how the data gets transformed.

Class imbalance is handled with class_weight="balanced", which tells Logistic Regression to weight the rare
"deteriorated" class more heavily during training, rather than resampling the data.

Repair spending per bridge is scaled like every other numeric feature.

In [9]:

import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def build_pipeline():
    """Preprocessing (impute + scale/encode) plus a balanced Logistic Regression."""
    numeric_pipeline = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, NUMERIC_COLS),
        ("cat", categorical_pipeline, CATEGORICAL_COLS),
    ])
    model = LogisticRegression(class_weight="balanced", max_iter=1000)
    return Pipeline([("preprocess", preprocessor), ("clf", model)])

In [10]:
pipeline = build_pipeline()
pipeline.fit(X_train, y_train)

n_encoded_features = len(pipeline.named_steps["preprocess"].get_feature_names_out())
print(f"Fit complete. {len(FEATURE_COLS)} raw features expand to {n_encoded_features} "
      f"after scaling and one-hot encoding.")

Fit complete. 27 raw features expand to 155 after scaling and one-hot encoding.


## 5. Evaluate performance on the 2025 test set

The test set is the 137,920 bridge transitions ending in 2025 that the model never trained on (see
Section 3), so these numbers reflect how the model would perform forecasting next year's risk for bridges it
already knows about.

Accuracy is not a useful headline metric here: only 4.5% of test rows actually deteriorated, so a model that
always predicts "no" would already be about 95% accurate while catching zero at-risk bridges. Instead this
section leads with ROC AUC and average precision (area under the precision-recall curve), which are not
fooled by the imbalance, then looks at precision and recall at the default 0.5 threshold, and finally checks
how well the model's risk ranking would work if it were used to prioritize which bridges get inspected first.

In [11]:
from sklearn.metrics import (
    average_precision_score, classification_report, confusion_matrix, roc_auc_score,
)

train_proba = pipeline.predict_proba(X_train)[:, 1]
test_proba = pipeline.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= 0.5).astype(int)

print(f"Train ROC AUC:            {roc_auc_score(y_train, train_proba):.3f}")
print(f"Test ROC AUC:             {roc_auc_score(y_test, test_proba):.3f}")
print(f"Train average precision:  {average_precision_score(y_train, train_proba):.3f}")
print(f"Test average precision:   {average_precision_score(y_test, test_proba):.3f}")
print(f"(Test positive rate, i.e. the average-precision floor a random ranking would score: "
      f"{y_test.mean():.3f})")

print("\nClassification report on the 2025 test set, at the default 0.5 probability threshold:")
print(classification_report(
    y_test, test_pred, target_names=["not deteriorated", "deteriorated"], digits=3
))

cm = confusion_matrix(y_test, test_pred)
cm_df = pd.DataFrame(
    cm,
    index=["actual: not deteriorated", "actual: deteriorated"],
    columns=["predicted: not deteriorated", "predicted: deteriorated"],
)
print("Confusion matrix:")
print(cm_df)

Train ROC AUC:            0.722
Test ROC AUC:             0.716
Train average precision:  0.111


Test average precision:   0.115
(Test positive rate, i.e. the average-precision floor a random ranking would score: 0.045)

Classification report on the 2025 test set, at the default 0.5 probability threshold:
                  precision    recall  f1-score   support

not deteriorated      0.975     0.666     0.792    131652
    deteriorated      0.084     0.642     0.148      6268

        accuracy                          0.665    137920
       macro avg      0.529     0.654     0.470    137920
    weighted avg      0.935     0.665     0.762    137920

Confusion matrix:
                          predicted: not deteriorated  predicted: deteriorated
actual: not deteriorated                        87731                    43921
actual: deteriorated                             2246                     4022


The classification report above is a single snapshot at the 0.5 threshold. Because class_weight="balanced"
was used, that threshold is not necessarily the right operating point for this problem — it treats a missed
deterioration and a false alarm as equally costly, which is unlikely to be true for a bridge inspection
program. The ROC and precision-recall curves below show the full tradeoff across every threshold at once.

In [12]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import precision_recall_curve, roc_curve

MODEL_COLOR = "#2a78d6"
REFERENCE_COLOR = "#c3c2b7"

fpr, tpr, _ = roc_curve(y_test, test_proba)
precision, recall, _ = precision_recall_curve(y_test, test_proba)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("ROC curve", "Precision-recall curve"),
)

fig.add_trace(
    go.Scatter(x=fpr, y=tpr, mode="lines", name="model", line=dict(color=MODEL_COLOR, width=2)),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="random guess",
               line=dict(color=REFERENCE_COLOR, width=2, dash="dash")),
    row=1, col=1,
)

fig.add_trace(
    go.Scatter(x=recall, y=precision, mode="lines", name="model", showlegend=False,
               line=dict(color=MODEL_COLOR, width=2)),
    row=1, col=2,
)
fig.add_trace(
    go.Scatter(x=[0, 1], y=[y_test.mean(), y_test.mean()], mode="lines", name="random guess",
               showlegend=False, line=dict(color=REFERENCE_COLOR, width=2, dash="dash")),
    row=1, col=2,
)

fig.update_xaxes(title_text="False positive rate", range=[0, 1], gridcolor="#e1e0d9", row=1, col=1)
fig.update_yaxes(title_text="True positive rate", range=[0, 1], gridcolor="#e1e0d9", row=1, col=1)
fig.update_xaxes(title_text="Recall", range=[0, 1], gridcolor="#e1e0d9", row=1, col=2)
fig.update_yaxes(title_text="Precision", range=[0, 1], gridcolor="#e1e0d9", row=1, col=2)
fig.update_layout(
    title_text="Model performance on the 2025 held-out test set",
    height=450, width=950,
    plot_bgcolor="#fcfcfb", paper_bgcolor="#fcfcfb",
    legend=dict(orientation="h", yanchor="bottom", y=1.08, xanchor="center", x=0.5),
)
fig.show()

### Capture rate: how useful is the model for prioritizing inspections?

BridgeWatch AI's actual use case is ranking bridges by risk, not making a one-off yes/no call, so the most
relevant test is: if inspectors worked down the model's risk ranking, how many of the bridges that truly
deteriorated in 2025 would they find within the first X% of bridges checked? This "capture rate" is a more
direct measure of real-world usefulness than any single classification metric above.

In [13]:
def capture_at_top_k(y_true, y_score, k_frac):
    """Share of actual positives found within the top k_frac of rows by predicted score."""
    y_true = np.asarray(y_true)
    n = len(y_true)
    k = max(1, int(np.ceil(n * k_frac)))
    top_k_idx = np.argsort(-y_score)[:k]
    return y_true[top_k_idx].sum() / y_true.sum(), k


y_test_arr = y_test.to_numpy()
n_positive = int(y_test_arr.sum())

print(f"{n_positive:,} of {len(y_test_arr):,} test-set bridges actually deteriorated in 2025.\n")
print(f"{'Top % of bridges by risk':<28}{'# bridges':>12}{'% of deteriorations found':>28}")
for k_frac in [0.05, 0.10, 0.20, 0.30, 0.50]:
    capture_rate, k = capture_at_top_k(y_test_arr, test_proba, k_frac)
    print(f"{int(k_frac * 100):>3}%{'':<24}{k:>12,}{capture_rate * 100:>27.1f}%")

6,268 of 137,920 test-set bridges actually deteriorated in 2025.

Top % of bridges by risk       # bridges   % of deteriorations found
  5%                               6,896                       17.6%
 10%                              13,792                       30.2%
 20%                              27,584                       47.7%
 30%                              41,376                       59.2%
 50%                              68,960                       78.2%


6. Do the weather and spending data help?

The model in Section 4 uses all three data sources. To see what each one adds, fit the same model three ways
and score each version on the same 2025 test set:

1. Bridge only: just the bridge inspection features.
2. Bridge + weather: adds the five weather features (research question 3).
3. Bridge + weather + spending: adds repair spending per bridge. This is the model from Section 4
   (research question 4).

Everything else stays the same: same preprocessing, same Logistic Regression settings, same train and test rows.
So any change in the scores comes from the added data, and nothing else. Each version is scored four ways: ROC AUC,
average precision, and the share of 2025 deteriorations found in the top 10% and top 20% of bridges by risk
(the capture rate from Section 5).

In [14]:
from sklearn.metrics import average_precision_score, roc_auc_score

WEATHER_COLS = ["tmax_mean_c", "tmin_mean_c", "tavg_mean_c", "prcp_total_mm", "freeze_thaw_days"]
SPENDING_COLS = ["repair_per_bridge_thousands"]
BRIDGE_NUMERIC_COLS = [c for c in NUMERIC_COLS if c not in WEATHER_COLS + SPENDING_COLS]


def build_pipeline_for(numeric_cols):
    """Same preprocessing and model as build_pipeline(), but with a chosen list of numeric columns."""
    numeric_pipeline = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, CATEGORICAL_COLS),
    ])
    model = LogisticRegression(class_weight="balanced", max_iter=1000)
    return Pipeline([("preprocess", preprocessor), ("clf", model)])


def score_model(y_true, y_score):
    """The four test-set scores used to compare model versions."""
    return {
        "ROC AUC": roc_auc_score(y_true, y_score),
        "Average precision": average_precision_score(y_true, y_score),
        "Found in top 10%": capture_at_top_k(y_true, y_score, 0.10)[0],
        "Found in top 20%": capture_at_top_k(y_true, y_score, 0.20)[0],
    }

In [15]:
versions = {
    "Bridge only": BRIDGE_NUMERIC_COLS,
    "Bridge + weather": BRIDGE_NUMERIC_COLS + WEATHER_COLS,
}

test_scores = {}
for name, numeric_cols in versions.items():
    cols = numeric_cols + CATEGORICAL_COLS
    version_pipeline = build_pipeline_for(numeric_cols)
    version_pipeline.fit(train_df[cols], y_train)
    test_scores[name] = version_pipeline.predict_proba(test_df[cols])[:, 1]

# The full model from Section 4 already has its test scores
test_scores["Bridge + weather + spending"] = test_proba

comparison = pd.DataFrame({name: score_model(y_test_arr, s) for name, s in test_scores.items()}).T
print("Scores on the 2025 test set (higher is better for all four):")
print(comparison.round(4))

Scores on the 2025 test set (higher is better for all four):
                             ROC AUC  Average precision  Found in top 10%  Found in top 20%
Bridge only                   0.7215             0.1174            0.3125            0.4884
Bridge + weather              0.7224             0.1175            0.3140            0.4885
Bridge + weather + spending   0.7164             0.1145            0.3019            0.4765


 How sure are we about these differences?

The differences between versions are small, so it is fair to ask whether they are real or just luck in which
bridges happened to land in the test set. A simple way to check is bootstrapping: redraw the test set at
random (with replacement) 200 times, recompute each version's ROC AUC on every redraw, and look at how much
the gain from adding each data source moves around. If the middle 95% of those gains stays on one side of zero,
the difference is very unlikely to be luck.

In [16]:
rng = np.random.default_rng(42)
n_test = len(y_test_arr)
gains = {"Adding weather": [], "Adding spending": []}

for _ in range(200):
    idx = rng.integers(0, n_test, n_test)
    auc = {name: roc_auc_score(y_test_arr[idx], s[idx]) for name, s in test_scores.items()}
    gains["Adding weather"].append(auc["Bridge + weather"] - auc["Bridge only"])
    gains["Adding spending"].append(auc["Bridge + weather + spending"] - auc["Bridge + weather"])

for name, values in gains.items():
    low, high = np.percentile(values, [2.5, 97.5])
    print(f"{name:<16} ROC AUC change: {np.mean(values):+.4f}  (95% of redraws between {low:+.4f} and {high:+.4f})")

Adding weather   ROC AUC change: +0.0010  (95% of redraws between +0.0004 and +0.0014)
Adding spending  ROC AUC change: -0.0060  (95% of redraws between -0.0066 and -0.0053)


The results split in two.

Weather helps a little, and the gain is real. Adding the weather features raises ROC AUC by about 0.001, and
every one of the middle 95% of redraws shows a gain. It is a small improvement, but it points the same way as
the EDA: harsher climates go with more deterioration (research question 3).

Spending makes the model worse, and that is also real. Adding spending per bridge lowers ROC AUC by about
0.006, and the model finds fewer of the bridges that actually deteriorated in its top 10% and 20%. The whole
95% range is below zero, so this is not bad luck. The next cell looks at why.

In [17]:
state_year = (df.groupby(["STATE", "from_year"])
              .agg(spending=("repair_per_bridge_thousands", "first"),
                   deterioration_rate=(TARGET, "mean"))
              .reset_index())

# Training years only: in years a state spent more than its usual, did it deteriorate more or less?
train_years = state_year[state_year["from_year"] < 2024].copy()
for col in ["spending", "deterioration_rate"]:
    train_years[col + "_vs_avg"] = train_years[col] - train_years.groupby("STATE")[col].transform("mean")
train_corr = train_years["spending_vs_avg"].corr(train_years["deterioration_rate_vs_avg"])
print(f"Within-state correlation in the training years (2020 to 2023): {train_corr:+.2f}")

# The test year: how did each state's 2024 spending and deterioration compare with its 2020 to 2023 average?
train_avg = train_years.groupby("STATE")[["spending", "deterioration_rate"]].mean()
year_2024 = state_year[state_year["from_year"] == 2024].set_index("STATE")
change = pd.DataFrame({
    "2024 spending vs usual ($K per bridge)": (year_2024["spending"] - train_avg["spending"]).round(1),
    "2024 deterioration vs usual (% points)": ((year_2024["deterioration_rate"]
                                                - train_avg["deterioration_rate"]) * 100).round(2),
})
print("\nHow 2024 compared with each state's 2020 to 2023 average:")
print(change)

Within-state correlation in the training years (2020 to 2023): +0.21

How 2024 compared with each state's 2020 to 2023 average:
       2024 spending vs usual ($K per bridge)  2024 deterioration vs usual (% points)
STATE                                                                                
CA                                        5.9                                   -0.57
FL                                        4.3                                   -1.26
MD                                       -5.9                                   -0.74
MI                                       13.5                                   -1.40
NY                                      -14.9                                    5.38
TX                                        9.4                                   -0.12
WA                                      -10.6                                   -0.11


This shows what went wrong. In the training years, years when a state spent more than usual had slightly
*more* deterioration (a correlation of about +0.2), so the model learned that higher spending means higher risk.
In 2024 that pattern turned around. New York spent about $15,000 less per bridge than usual, and its
deterioration jumped by more than 5 percentage points. Michigan spent the most extra and saw its deterioration
fall. The model used the rule it learned, scored New York's bridges as *lower* risk in exactly the year they got
worse, and ranked them badly.

The underlying problem is that spending has only 35 values (7 states by 5 years), and the model learns from just
28 of them. That is not enough to find a pattern that holds from one year to the next, and this one did not.

What this means for the project:

* Research question 4: in this data, state bridge spending does not help predict which bridges will
  deteriorate. Across states, higher spending goes with more deterioration because states spend where bridges
  are already in poor shape (EDA Section 9). Within states, the year to year link is too weak and unstable to
  forecast with. That is a real finding, even though it is not the result one might have hoped for.
* The final model: the bridge + weather version scores best, so it is the better choice for the app.
  Spending can still be shown in the app as background about each state, just not used to score bridges.